In [4]:
import math
import numpy as np
from scipy.special import kolmogorov

# Набор исходных данных
N = 100
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
alpha = 0.05

def F_norm(x, mean, sigma):
    """Функция нормального распределения."""
    return 0.5 * (1 + math.erf((x - mean) / (np.sqrt(2) * sigma)))

# a) Колмогоров для равномерного распределения


F_emp = np.array([sum(m[:i]) for i in range(len(m) + 1)]) / N
F_th = np.arange(10) / 10


delta = np.sqrt(N) * np.max([
    max(np.abs(F_th[i] - F_emp[i]), np.abs(F_th[i] - F_emp[i + 1]))
    for i in range(F_th.size)
])


p_value = kolmogorov(delta)


p_value_str = f"{p_value:.6f}" if isinstance(p_value, float) else str(p_value)

print("a)")
print(f"p-value по критерию Колмогорова: {p_value_str:>20} ")


if p_value < alpha:
    print("H0 отвергается")
else:
    print("нет оснований отвергать H0")


# b) Колмогоров для нормального распределения


segments = np.array([(-np.inf, 1)] + [(i, i + 1) for i in range(1, 9)] + [(9, np.inf)])
sample = np.repeat(np.arange(len(m)), m)
alpha = np.mean(sample)
sigma = np.sqrt(np.var(sample) * N / (N - 1))

def F_norm_wave(x):
    return F_norm(x, alpha, sigma)

P = [F_norm_wave(i[1]) - F_norm_wave(i[0]) for i in segments]

delta_b = sum((N * P[i] - m[i]) ** 2 / (N * P[i]) for i in range(10))


print("\nΔ =", delta_b)


bootstrap_iterations = 50000
bootstrap_delta = []

x = np.arange(10)
delta_wave = np.sqrt(N) * np.max([
    max(np.abs(F_norm_wave(x[i]) - F_emp[i]), np.abs(F_norm_wave(x[i]) - F_emp[i + 1]))
    for i in x
])

for _ in range(bootstrap_iterations):
    random_sample = np.array(sorted(np.random.normal(alpha, sigma, N)))
    alpha_bootstrap = random_sample.mean()
    sigma_bootstrap = np.sqrt(random_sample.var() * N / (N - 1))

    F_bootstrap_emp = [i / N for i in range(N + 1)]

    def F_bootstrap_wave(j):
        return F_norm(random_sample[j], alpha_bootstrap, sigma_bootstrap)

    sup = np.max([
        max(np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j]),
            np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j + 1]))
        for j in range(len(random_sample))
    ])

    bootstrap_delta.append(np.sqrt(N) * sup)

bootstrap_delta = np.array(bootstrap_delta)
p_value_bootstrap = len(bootstrap_delta[bootstrap_delta >= delta_wave]) / bootstrap_iterations


p_value_str_bs = f"{p_value_bootstrap:.6f}"
print("\nb)")
print(f"p-value: {p_value_str_bs:<25} ")


if p_value_bootstrap < alpha:
    print("H0 отвергается")
else:
    print("нет оснований отвергать H0")


print("\nΔ = {:.6f}".format(delta_b))

a)
p-value по критерию Колмогорова:             0.039682 
H0 отвергается

Δ = 16.87106704806872

b)
p-value: 0.014500                  
H0 отвергается

Δ = 16.871067
